# 01 · Подготовка среды и загрузка OmniDocBench

Этот ноутбук готовит Colab-среду к экспериментам:
1. Устанавливает зависимости из `requirements.txt`.
2. Скачивает датасет **OmniDocBench v1.6** (`opendatalab/OmniDocBench` на HuggingFace).
3. Загружает разметку, фильтрует страницы типа `academic_literature` (научные статьи arXiv-стиля).
4. Показывает превью первой страницы и её ground-truth.

## 1. Установка зависимостей

In [ ]:
# В Colab можно использовать GPU runtime → T4/L4/A100
!nvidia-smi || echo 'GPU не подключена'

In [ ]:
!pip install -q -r requirements.txt

## 2. Bootstrap путей

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, subprocess, pathlib

REPO_NAME = "ocr_eval"
if not pathlib.Path(REPO_NAME).exists():
    # Замените URL на ваш форк, если работаете в Colab
    # !git clone https://github.com/<your-username>/ocr_eval.git
    pass

if pathlib.Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("CWD =", os.getcwd())


## 3. Скачивание OmniDocBench

In [ ]:
from src.dataset_loader import download_omnidocbench, load_omnidocbench, iter_pages

DATA_ROOT = 'data/OmniDocBench'
download_omnidocbench(DATA_ROOT, source='huggingface')
print('OK, датасет в', DATA_ROOT)

## 4. Фильтр научных страниц и быстрый превью

In [ ]:
# Берём 100 случайных страниц-научных публикаций для экспериментов
items = load_omnidocbench(
    root=DATA_ROOT,
    page_types=['academic_literature'],
    languages=['english'],
    subset_size=100,
    seed=42,
)
print(f'Отобрано страниц: {len(items)}')
items[0].to_dict() if items else 'empty'

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

first = items[0]
img = Image.open(Path(DATA_ROOT) / first.image_path).convert('RGB')
fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(img); ax.axis('off')
ax.set_title(f'{first.page_id}  ·  {first.page_type}')
plt.show()

## 5. Сохраняем список выбранных страниц

Все 4 ноутбука инференса используют один и тот же subset — так результаты сравнимы.

In [ ]:
import json, pathlib
subset_path = pathlib.Path('data/subset.json')
subset_path.parent.mkdir(parents=True, exist_ok=True)
subset_path.write_text(
    json.dumps([x.to_dict() for x in items], ensure_ascii=False),
    encoding='utf-8',
)
print('сохранено:', subset_path, subset_path.stat().st_size, 'байт')

---
*Готово.* Дальше — `02_deepseek_ocr.ipynb`.